# USA Housing EDA

    This notebook is intended to look more like an industry-style exploratory analysis:
    it profiles the dataset, surfaces data quality issues, inspects target behavior,
    reviews feature distributions, and turns findings into concrete pipeline decisions.


## Questions This Notebook Should Answer

    - Is the dataset physically available to the running kernel?
    - What is the schema, row count, null profile, and duplicate profile?
    - What does the target distribution look like, including suspicious values?
    - Which features are likely useful, skewed, sparse, constant, or risky?
    - What exact validation and transformation rules should move into production code?


In [ ]:
from __future__ import annotations

    import warnings
    from pathlib import Path

    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd

    try:
        import seaborn as sns
    except ImportError:
        sns = None

    warnings.filterwarnings("ignore")
    plt.style.use("seaborn-v0_8-whitegrid")
    pd.set_option("display.max_columns", 100)
    pd.set_option("display.width", 140)

    DATA_RELATIVE_PATH = Path("data/raw/usa-housing-dataset/USA Housing Dataset.csv")

    def is_repo_root(candidate: Path) -> bool:
        return (
            (candidate / "main.py").exists()
            and (candidate / "src").exists()
            and (candidate / "config").exists()
        )

    def resolve_data_path() -> tuple[Path | None, Path | None, str]:
        cwd = Path.cwd().resolve()
        candidates = [cwd, *cwd.parents]
        candidates += [
            Path.home(),
            Path.home() / "pranav",
            Path.home() / "pranav" / "house-price-ml",
            Path("/content"),
            Path("/content/drive/MyDrive"),
        ]

        seen = set()
        for candidate in candidates:
            try:
                candidate = candidate.resolve()
            except FileNotFoundError:
                continue
            if candidate in seen:
                continue
            seen.add(candidate)
            if is_repo_root(candidate):
                data_path = candidate / DATA_RELATIVE_PATH
                if data_path.exists():
                    return candidate, data_path, "repo-relative"

        search_roots = [cwd, Path.home(), Path("/content")]
        for root in search_roots:
            if not root.exists():
                continue
            try:
                matches = list(root.rglob("USA Housing Dataset.csv"))
            except Exception:
                continue
            for match in matches:
                if "usa-housing-dataset" in str(match.parent):
                    return None, match.resolve(), "recursive-search"

        return None, None, "not-found"

    REPO_ROOT, DATA_PATH, RESOLUTION_MODE = resolve_data_path()

    print("Kernel cwd:", Path.cwd())
    print("Resolution mode:", RESOLUTION_MODE)
    print("Repo root:", REPO_ROOT)
    print("Dataset path:", DATA_PATH)

    if DATA_PATH is None or not DATA_PATH.exists():
        raise FileNotFoundError(
            "Dataset file not found. If you are using a Google Colab kernel from VS Code, "
            "that remote kernel cannot see your local repo files. Use a local kernel, or upload/clone the repo into Colab."
        )


In [ ]:
df = pd.read_csv(DATA_PATH)
    df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d %H:%M:%S", errors="coerce")

    print("shape:", df.shape)
    print("columns:", list(df.columns))
    df.head(3)


## 1. Dataset Snapshot

    Start with a compact dataset profile: shape, dtypes, nulls, uniqueness, and duplicate rate.
    This is the minimum baseline before model work.


In [ ]:
profile = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "unique_count": df.nunique(dropna=False),
        "sample_value": [df[col].iloc[0] for col in df.columns],
    }).sort_index()

    duplicate_rows = int(df.duplicated().sum())
    row_count, column_count = df.shape

    print("row_count:", row_count)
    print("column_count:", column_count)
    print("duplicate_rows:", duplicate_rows)
    profile


In [ ]:
numeric_columns = [
        "price", "bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors",
        "waterfront", "view", "condition", "sqft_above", "sqft_basement",
        "yr_built", "yr_renovated",
    ]
    categorical_columns = ["street", "city", "statezip", "country"]

    summary = df[numeric_columns].describe().T
    summary["skew"] = df[numeric_columns].skew().round(3)
    summary["missing_pct"] = (df[numeric_columns].isna().mean() * 100).round(2)
    summary


## 2. Target Analysis

    The target deserves dedicated attention. We need to know its scale, tail behavior,
    invalid values, and whether a log transform is likely to help.


In [ ]:
target = df["price"]
    non_positive_price_count = int((target <= 0).sum())
    print("price_min:", float(target.min()))
    print("price_median:", float(target.median()))
    print("price_max:", float(target.max()))
    print("non_positive_price_count:", non_positive_price_count)

    df.loc[target <= 0, ["date", "price", "bedrooms", "bathrooms", "sqft_living", "city"]].head(10)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    axes[0].hist(df["price"], bins=50, color="#4C78A8", edgecolor="white")
    axes[0].set_title("Price Distribution")
    axes[0].set_xlabel("price")

    axes[1].boxplot(df["price"], vert=False)
    axes[1].set_title("Price Boxplot")
    axes[1].set_xlabel("price")

    positive_price = df.loc[df["price"] > 0, "price"]
    axes[2].hist(np.log1p(positive_price), bins=50, color="#F58518", edgecolor="white")
    axes[2].set_title("log1p(Price) for Positive Targets")
    axes[2].set_xlabel("log1p(price)")

    plt.tight_layout()
    plt.show()


## 3. Feature Distribution Review

    Review continuous features for scale, skew, and outliers. This informs winsorization,
    log transforms, scaling decisions, and robustness expectations.


In [ ]:
continuous_features = [
        "sqft_living", "sqft_lot", "sqft_above", "sqft_basement",
        "bedrooms", "bathrooms", "floors",
    ]

    fig, axes = plt.subplots(3, 3, figsize=(16, 12))
    axes = axes.flatten()

    for ax, column in zip(axes, continuous_features):
        ax.hist(df[column], bins=40, color="#54A24B", edgecolor="white")
        ax.set_title(column)
        ax.set_xlabel(column)

    for ax in axes[len(continuous_features):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
skew_table = pd.DataFrame({
        "skew": df[continuous_features + ["price"]].skew().round(3),
        "min": df[continuous_features + ["price"]].min(),
        "max": df[continuous_features + ["price"]].max(),
    }).sort_values("skew", key=lambda s: s.abs(), ascending=False)

    skew_table


## 4. Categorical and Geographic Review

    Understand categorical cardinality and geographic concentration. This informs encoding,
    rare-category treatment, and leakage concerns.


In [ ]:
categorical_profile = pd.DataFrame({
        "unique_count": df[categorical_columns].nunique(),
        "top_value": [df[col].mode().iloc[0] for col in categorical_columns],
        "top_freq": [int(df[col].value_counts().iloc[0]) for col in categorical_columns],
    })

    categorical_profile


In [ ]:
top_cities = df["city"].value_counts().head(15)

    fig, ax = plt.subplots(figsize=(12, 5))
    top_cities.sort_values().plot(kind="barh", ax=ax, color="#B279A2")
    ax.set_title("Top 15 Cities by Row Count")
    ax.set_xlabel("count")
    plt.tight_layout()
    plt.show()

    print("country value counts:
", df["country"].value_counts())
    print("
statezip prefixes:
", df["statezip"].str.split().str[0].value_counts().head(10))


## 5. Temporal Review

    Even if the model later drops raw timestamps, time fields should still be examined for
    coverage, seasonality hints, and parseability.


In [ ]:
date_summary = pd.DataFrame({
        "min_date": [df["date"].min()],
        "max_date": [df["date"].max()],
        "null_dates": [int(df["date"].isna().sum())],
        "unique_dates": [int(df["date"].nunique())],
    })

    date_summary


In [ ]:
monthly_counts = df.assign(month=df["date"].dt.to_period("M").astype(str))["month"].value_counts().sort_index()
    monthly_median_price = df.assign(month=df["date"].dt.to_period("M").astype(str)).groupby("month")["price"].median()

    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    monthly_counts.plot(kind="bar", ax=axes[0], color="#4C78A8")
    axes[0].set_title("Rows by Month")
    axes[0].set_xlabel("month")
    axes[0].set_ylabel("count")

    monthly_median_price.plot(marker="o", ax=axes[1], color="#E45756")
    axes[1].set_title("Median Price by Month")
    axes[1].set_xlabel("month")
    axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()


## 6. Correlation and Relationship Review

    Correlation is not a feature-selection rule by itself, but it is useful for triage and
    interaction intuition.


In [ ]:
correlation = df[numeric_columns].corr(numeric_only=True)
    correlation["price"].sort_values(ascending=False).to_frame("corr_with_price")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
    if sns is not None:
        sns.heatmap(correlation, cmap="coolwarm", center=0, ax=ax)
    else:
        heatmap = ax.imshow(correlation, cmap="coolwarm", aspect="auto")
        fig.colorbar(heatmap, ax=ax)
        ax.set_xticks(range(len(correlation.columns)))
        ax.set_xticklabels(correlation.columns, rotation=90)
        ax.set_yticks(range(len(correlation.index)))
        ax.set_yticklabels(correlation.index)
    ax.set_title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()


## 7. Explicit Data Quality and Trainability Checks

    This section mirrors what should become production validation or transformation rules.
    The goal is to turn notebook observations into enforceable contracts.


In [ ]:
quality_checks = {
        "row_count": int(len(df)),
        "column_count": int(df.shape[1]),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_any": bool(df.isna().any().any()),
        "non_positive_price": int((df["price"] <= 0).sum()),
        "non_positive_sqft_living": int((df["sqft_living"] <= 0).sum()),
        "non_positive_sqft_lot": int((df["sqft_lot"] <= 0).sum()),
        "non_positive_sqft_above": int((df["sqft_above"] <= 0).sum()),
        "non_positive_yr_built": int((df["yr_built"] <= 0).sum()),
        "country_not_usa": int((df["country"] != "USA").sum()),
        "state_prefix_not_wa": int((df["statezip"].str.split().str[0] != "WA").sum()),
        "invalid_date_parse": int(df["date"].isna().sum()),
    }

    pd.Series(quality_checks, name="value")


In [ ]:
def iqr_outlier_summary(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
        rows = []
        for column in columns:
            q1 = frame[column].quantile(0.25)
            q3 = frame[column].quantile(0.75)
            iqr = q3 - q1
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            count = int(((frame[column] < lower) | (frame[column] > upper)).sum())
            rows.append({
                "column": column,
                "lower_bound": round(float(lower), 3),
                "upper_bound": round(float(upper), 3),
                "outlier_count": count,
            })
        return pd.DataFrame(rows).sort_values("outlier_count", ascending=False)

    iqr_outlier_summary(df, ["price", "sqft_living", "sqft_lot", "bedrooms", "bathrooms"])


## 8. Recommended Production Decisions

    The notebook is only useful if it drives hard decisions in the pipeline.
    These are the concrete takeaways from the current dataset snapshot.

    **Validation decisions**
    - Keep schema, null, duplicate, numeric, and datetime validation as hard checks.
    - Keep `sqft_living`, `sqft_lot`, `sqft_above`, and `yr_built` as strictly positive hard checks.
    - Keep non-positive `price` as a warning at validation time, not a hard failure, because the current dataset contains them.
    - Keep `country=USA` as a warning-style contract check.

    **Transformation decisions**
    - Drop or quarantine rows with non-positive `price` before model training.
    - Consider log-transforming `price`, `sqft_living`, and `sqft_lot` because of strong right skew.
    - Split `date` into derived calendar features or drop it after feature engineering.
    - Treat `street` carefully because it has high cardinality and weak generalization potential.
    - Prefer `city` and parsed ZIP/state features over raw full street strings for a baseline model.

    **Modeling implications**
    - Start with a baseline on cleaned numeric + medium-cardinality categorical features.
    - Compare a linear baseline against tree-based regressors after transformation is implemented.
    - Track MAE, RMSE, and R2, and compare both raw-target and log-target training.
